### Function imports

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cf
import uxarray as ux
import intake
import numpy as np

import pandas as pd

import healpix as hp
import holoviews as hv

import easygems.healpix as egh
import easygems.remap as egr

import easygems.healpix as eghp

import cmocean
import geoviews.feature as gf


sigma = 5.67E-8  # Stefan-Boltzmann constant (W/m^2/K^4)

### Define the catalog

In [2]:
node_id = 'NCAR'
cat = intake.open_catalog("https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml")[node_id]

In [3]:
list(cat)

['CERES_EBAF',
 'ERA5',
 'IR_IMERG',
 'JRA3Q',
 'MERRA2',
 'arp-gem-1p3km',
 'arp-gem-2p6km',
 'casesm2_10km_nocumulus',
 'ew_dyamond3_2D',
 'ew_dyamond3_3D',
 'icon_d3hp003',
 'icon_d3hp003aug',
 'icon_d3hp003feb',
 'icon_ngc4008',
 'ifs_tco3999-ng5_deepoff',
 'ifs_tco3999-ng5_rcbmf',
 'ifs_tco3999-ng5_rcbmf_cf',
 'ifs_tco3999_rcbmf',
 'mpas_dyamond1',
 'mpas_dyamond2',
 'mpas_dyamond3',
 'nicam_220m_test',
 'nicam_gl11',
 'scream-dkrz',
 'scream2D_hrly',
 'scream_lnd',
 'scream_ne120',
 'tracking-d3hp003',
 'um_Africa_km4p4_RAL3P3_n1280_GAL9_nest',
 'um_CTC_km4p4_RAL3P3_n1280_GAL9_nest',
 'um_SAmer_km4p4_RAL3P3_n1280_GAL9_nest',
 'um_SEA_km4p4_RAL3P3_n1280_GAL9_nest',
 'um_glm_n1280_CoMA9_TBv1p2',
 'um_glm_n1280_GAL9',
 'um_glm_n2560_RAL3p3',
 'wrf_conus',
 'wrf_samerica']

In [4]:
# Query a data set
pd.DataFrame(cat['ew_dyamond3_2D'].describe()["user_parameters"])

,name,description,type,allowed,default
0,time,temporal resolution of the dataset,str,"[PT1H, PT3H]",PT1H
1,zoom,zoom resolution of the dataset,int,"[9, 8, 7, 6, 5, 4, 3, 2, 1]",7


### Define the zoom level

In [5]:
time_slice = slice("2019-01-01", "2021-03-01")

In [6]:
zoom_lev = 8

In [7]:
#tmp = cat['ew_dyamond3_2D'](zoom=zoom_lev,time='PT1H').to_dask()
#tmp

### Read in data into UxArray (precipitation)

In [8]:
if node_id=='NERSC':
    scl_icon = 86400. #kg/m2/s to mm/day
    ds_icon = cat['icon_d3hp003'](zoom=zoom_lev).to_dask()
    uxds_icon = ux.UxDataset.from_healpix(ds_icon)
    uxda_pr_icon = uxds_icon['pr'].sel(time=time_slice) #*scl_icon # mm/day

    scl_scream = 86400.*1000. #m/s to mm/day
    ds_scream = cat['scream2D_hrly'](zoom=zoom_lev).to_dask()
    uxds_scream = ux.UxDataset.from_healpix(ds_scream)
    uxda_pr_scream = uxds_scream['pr'].sel(time=time_slice)# *scl_scream # mm/day

    scl_nicam = 86400. #kg/m2/s to mm/day
    ds_nicam = cat['nicam_gl11'](zoom=zoom_lev, time='PT3H').to_dask()
    uxds_nicam = ux.UxDataset.from_healpix(ds_nicam)
    uxds_nicam.assign_coords(hour=uxds_nicam.time.dt.hour)
    uxds_nicam.assign_coords(month=uxds_nicam.time.dt.month)
    uxda_pr_nicam = uxds_nicam['pr'].sel(time=time_slice) #*scl_nicam # mm/day

#scl_mpas = 48. #mm/30min to mm/day
#ds_mpas = cat['mpas_dyamond3'](zoom=zoom_lev, time='PT30M').to_dask()
#uxds_mpas = ux.UxDataset.from_healpix(ds_mpas)
#uxda_pr_mpas = uxds_mpas['rainnc'].sel(time=time_slice) #*scl_nicam # mm/day

    scl_um = 86400. #m/s to mm/day
    pth='/global/cfs/cdirs/m4581/gsharing/hackathon/UM/glm.n2560_RAL3p3/'
    fn=pth+'data.healpix.PT1H.z'+str(zoom_lev)+'.zarr'
    ds_um=xr.open_dataset(fn)
    uxds_um = ux.UxDataset.from_healpix(ds_um)
    uxds_um.assign_coords(hour=uxds_um.time.dt.hour)
    uxds_um.assign_coords(month=uxds_um.time.dt.month)
    uxda_pr_um = uxds_um['pr'].sel(time=time_slice) #*scl_um # mm/day

# scl_imerg = 24. #mm/hr to mm/day
# uxds_imerg = ux.UxDataset.from_healpix('/glade/derecho/scratch/andrew/hackathon/IMERG_V07B_hp9.zarr')
# scl_imerg = 24. #mm/hr to mm/day
# ds_imerg = cat['IR_IMERG'](zoom=9).to_dask()
# uxds_imerg = ux.UxDataset.from_healpix(ds_imerg)

# uxda_pr_imerg_fine = uxds_imerg['precipitation'].sel(time=time_slice)
# uxda_pr_imerg_fine = uxda_pr_imerg_fine.chunk({'time': 48, 'n_face': uxda_pr_imerg_fine.sizes['n_face']})

# level_down = 9 - zoom_lev
# uxda_pr_imerg_tmp = uxda_pr_imerg_fine
# for i in range(level_down):
#     uxda_pr_imerg_tmp = uxda_pr_imerg_tmp.coarsen(n_face=4).mean()
#     uxda_pr_imerg_tmp['crs'].attrs['healpix_nside'] = 2**int(9 - i - 1)
# uxda_pr_imerg = uxda_pr_imerg_tmp

# # level_down = 9 - zoom_lev
# # for i in range(level_down):
# #     uxds_imerg = uxds_imerg.coarsen(n_face=4).mean()
# #     uxds_imerg['crs'].attrs['healpix_nside'] = 2**int(9 - i - 1) #hp_nside // 2
# # uxda_rlut_imerg_rename = uxds_imerg.rename({'n_face': 'cell'})
# # uxds_imerg = ux.UxDataset.from_healpix(uxda_rlut_imerg_rename)

# # uxda_pr_imerg_orig = uxds_imerg['precipitation'].sel(time=time_slice)
# # uxda_pr_imerg_orig = uxda_pr_imerg_orig.chunk({'time': 48, 'n_face': uxda_pr_imerg_orig.sizes['n_face']})
# uxda_pr_imerg = uxda_pr_imerg.resample(time='1H').first().compute() #*scl_imerg # mm/day

    scl_cas = 24*1000. #assume m/hr
    ds_cas = cat['casesm2_10km_nocumulus'](zoom=zoom_lev,time='PT1H').to_dask()
    uxds_cas = ux.UxDataset.from_healpix(ds_cas)
    uxds_cas.assign_coords(hour=uxds_cas.time.dt.hour)
    uxds_cas.assign_coords(month=uxds_cas.time.dt.month)
    uxda_pr_cas = uxds_cas['pr'].sel(time=time_slice) #*scl_cas # mm/day


In [9]:
if node_id=='NCAR':
    scl_ew = 86400. #kg/m2/s to mm/day
    ds_ew = cat['ew_dyamond3_2D'](zoom=zoom_lev,time='PT1H').to_dask()
    uxds_ew = ux.UxDataset.from_healpix(ds_ew)
    uxds_ew.assign_coords(hour=uxds_ew.time.dt.hour)
    uxds_ew.assign_coords(month=uxds_ew.time.dt.month)
    uxda_pr_ew = uxds_ew['pr'].sel(time=time_slice) #*scl_ew # mm/day

# Need to figure out right MPAS scaling....
    scl_mpas = 1/48. #mm/30min to mm/day
    ds_mpas = cat['mpas_dyamond3'](zoom=zoom_lev, time='PT30M').to_dask()
    uxds_mpas = ux.UxDataset.from_healpix(ds_mpas)
    uxda_pr_mpas = uxds_mpas['rainnc'].sel(time=time_slice) #*scl_nicam # mm/day

    scl_scream = 86400.*1000. #m/s to mm/day
    ds_scream = cat['scream2D_hrly'](zoom=zoom_lev).to_dask()
    uxds_scream = ux.UxDataset.from_healpix(ds_scream)
    uxda_pr_scream = uxds_scream['pr'].sel(time=time_slice)# *scl_scream # mm/day

# scl_imerg = 24. #mm/hr to mm/day
# uxds_imerg = ux.UxDataset.from_healpix('/glade/derecho/scratch/andrew/hackathon/IMERG_V07B_hp9.zarr')
# scl_imerg = 24. #mm/hr to mm/day
# ds_imerg = cat['IR_IMERG'](zoom=9).to_dask()
# uxds_imerg = ux.UxDataset.from_healpix(ds_imerg)


/glade/u/apps/opt/conda/envs/2025-digital-earths-global-hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),
/glade/u/apps/opt/conda/envs/2025-digital-earths-global-hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),
/glade/u/apps/opt/conda/envs/2025-digital-earths-global-hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of

### Test plot the data (zonal mean precip) 
Warning: even at zoom = 8 this blows up memory!

In [10]:
if (zoom_lev < 6):
    if node_id=='NERSC':
        zm_cas=uxda_pr_cas.mean(dim='time').zonal_mean()*scl_cas
        zm_um=uxda_pr_um.mean(dim='time').zonal_mean()*scl_um
        zm_icon=uxda_pr_icon.mean(dim='time').zonal_mean()*scl_icon
        zm_nicam=uxda_pr_nicam.mean(dim='time').zonal_mean()*scl_nicam
        zm_scream=uxda_pr_scream.mean(dim='time').zonal_mean()*scl_scream

        plt.plot(zm_cas.latitudes,zm_cas,label='CAS')
        plt.plot(zm_um.latitudes,zm_um,label='UM')
        plt.plot(zm_icon.latitudes,zm_icon,label='ICON')
        plt.plot(zm_nicam.latitudes,zm_nicam,label='NICAM')
        plt.plot(zm_scream.latitudes,zm_scream,label='SCREAM')
        plt.legend()
    if node_id=='NCAR':
        zm_ew=uxda_pr_ew.mean(dim='time').zonal_mean()*scl_ew
        zm_mpas=uxda_pr_mpas.mean(dim='time').zonal_mean()*scl_mpas
        zm_scream=uxda_pr_scream.mean(dim='time').zonal_mean()*scl_scream
        
        plt.plot(zm_ew.latitudes,zm_ew,label='EarthWorks')
        plt.plot(zm_mpas.latitudes,zm_mpas,label='MPAS')
        plt.plot(zm_scream.latitudes,zm_scream,label='SCREAM')
        plt.legend()
        

### Frequency

In [11]:
freq = False

month = 2 # February !!

threshold = 1 # mm/day

if freq:
    #frequency_scream = (uxda_pr_scream > threshold / scl_scream).groupby("time.month").mean("time").sel(month=month).compute()
    #frequency_icon = (uxda_pr_icon > threshold / scl_icon).groupby("time.month").mean("time").sel(month=month).compute()
    #frequency_nicam = (uxda_pr_nicam > threshold / scl_nicam).groupby("time.month").mean("time").sel(month=month).compute()
    frequency_um = (uxda_pr_um > threshold / scl_um).groupby("time.month").mean("time").sel(month=month).compute()
    # frequency_imerg = (uxda_pr_imerg > threshold / scl_imerg).groupby("time.month").mean("time").sel(month=month).compute()

In [12]:
### Make all months for UM
if freq:
    threshold = 1
    frequency_um_all = (uxda_pr_um > threshold / scl_um).groupby("time.month").mean("time").compute()

In [13]:
### Write to NetCDF
if freq:
    frequency_um_all.to_netcdf("frequency_gt_1mm_um_z"+str(zoom_lev)+"_v3.nc")

### Intensity: Juan Version

In [14]:
## bins from 1 to 500 mm/day
#bins = np.logspace(np.log10(1), np.log10(500), 21)
##bins
#binctrs = 0.5 * (bins[:-1] + bins[1:])
#binctrs

### Define Bins 
(with some help from Claude)

Prompt: I am using python and xarray. I want to first figure out the edges of a set of bins. Assume bin centers are : binctr=[0.1,0.2,0.5,1,2,5,10,20,50,100,200,300,500,600,800.,1000.] and the edge of the first bin can be 0 and the last bin can be 2000. Find the edges between the bins in log space in a python list please.



In [15]:
# Bin centers
binctr = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200, 300, 500, 600, 800, 1000]

# Create bin edges list
bin_edges = [0.]  # First edge is 0 as specified

# Calculate intermediate edges as geometric means between bin centers
for i in range(len(binctr)-1):
    # Geometric mean: sqrt(a * b)
    edge = np.sqrt(binctr[i] * binctr[i+1])
    bin_edges.append(edge)

# Add the final edge
bin_edges.append(2000.)  # Last edge is 2000 as specified

#print(bin_edges)

bins=bin_edges
binctrs=binctr

#### Select Model Here

In [23]:
del prate

In [24]:
prate = uxda_pr_ew * scl_ew

#### Histogram based function
Needs some chunking to be improved: uses 33GB at level 7 (40s), but blows memory at NERSC for level 8 (200GB) on a shared node. Seems okay on a dedicated node (100GB, 1m30s)

In [25]:
# dens control if density or counts returned, not
# sure which is better here... maybe dens due to 
# different 1H, 3H output frequencies.??
def histogram_per_x(values, bins, dens):
    hist, _ = np.histogram(values, bins=bins, density=dens)
    return hist

dens = False

# get error witouth this line
prate = prate.chunk({"time": -1})  

prate.coords["month"] = prate["time"].dt.month.compute()

monthly_hist = xr.apply_ufunc(
    histogram_per_x,
    prate.groupby("month"),
    input_core_dims=[["time"]],
    output_core_dims=[["bin"]],
    output_sizes={"bin": len(bins) - 1},
    kwargs={"bins": bins,
            "dens": dens}, # make dens False for counts
    vectorize=True,
    dask="parallelized",
    dask_gufunc_kwargs={"allow_rechunk": True},
    #output_dtypes=[int]
)


/glade/derecho/scratch/andrew/tmp/ipykernel_84218/664420445.py:15: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  monthly_hist = xr.apply_ufunc(


In [26]:
%%time
monthly_hist_sel = monthly_hist.compute()
#max_val = monthly_hist_sel.max().item()

CPU times: user 2min 47s, sys: 27.8 s, total: 3min 15s
Wall time: 3min 56s


In [27]:
monthly_hist_sel=monthly_hist_sel.assign_coords(bin=binctrs)
#monthly_hist_sel

In [28]:
# Write to NetCDF
write = True
if write:
    monthly_hist_sel.to_netcdf("intensity_ew_z"+str(zoom_lev)+"_v3.nc")

In [ ]:
### Plotting

In [ ]:
import matplotlib.colors as mcolors
#Plot
bn = 10
month = 7

# if density 
if dens == True: 
    norm = mcolors.LogNorm(vmin=1e-3, vmax=1e-2)
    label='Density'
else:
    norm = mcolors.Normalize(vmin=0, vmax=600)
    label='Counts'


title = f"Bin: {bins[bn]:.2f} – {bins[bn+1]:.2f} mm/day"


projection = ccrs.Robinson(central_longitude=-85)



fig, ax = plt.subplots(
    figsize=(8, 4),
    subplot_kw={"projection": projection}, 
    constrained_layout=True
)

ax.set_global()
P = egh.healpix_show(monthly_hist_sel[month,:,bn], ax=ax, norm=norm)
fig.colorbar(P, label=label, shrink=0.5, orientation="horizontal", pad=0.05)
ax.coastlines(color="gray", lw=1)
plt.title(title)

In [ ]:
plt.plot(binctrs,monthly_hist_sel[month,200,:])

In [ ]:
plt.plot(prate[:,200].values)

### Intensity: Andrew Version

In [ ]:
# Find max
(uxda_pr_um*scl_um).max()

### Define Bins 
(with some help from Claude)

Prompt: I am using python and xarray. I want to first figure out the edges of a set of bins. Assume bin centers are : binctr=[0.1,0.2,0.5,1,2,5,10,20,50,100,200,300,500,600,800.,1000.] and the edge of the first bin can be 0 and the last bin can be 2000. Find the edges between the bins in log space in a python list please.



In [ ]:
# Bin centers
binctr = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200, 300, 500, 600, 800, 1000]

# Create bin edges list
bin_edges = [0.]  # First edge is 0 as specified

# Calculate intermediate edges as geometric means between bin centers
for i in range(len(binctr)-1):
    # Geometric mean: sqrt(a * b)
    edge = np.sqrt(binctr[i] * binctr[i+1])
    bin_edges.append(edge)

# Add the final edge
bin_edges.append(2000.)  # Last edge is 2000 as specified

print(bin_edges)

In [29]:
# Deleted the rest: doesn't work. 